# Stokes–Brinkman: two levels of error and two adaptive strategies

L14's Algorithm 1 refines macroelements from $\eta_K=\eta_{2,K}+\sum_F\eta_{1,F}$.
Algorithm 2 marks macrofaces, bisects their maximal-indicator segments, and refines
neighboring local meshes when the local residual dominates. These are distinct
approximation histories, with the same Stokes–Brinkman estimator.

The polynomial test uses $\psi=128x^2(x-1)^2y^2(y-1)^2$,
$u=(\partial_y\psi,-\partial_x\psi)$ and $p=150(x-1/2)(y-1/2)$.
This explicitly solenoidal convention avoids the incompatible printed component
ordering. Viscosities 1 and 0.01, skeletal degrees 0–2 and five uniform levels are
recorded. The figures compare individual components with the printed tables;
identical historical meshes and inverse constants are not established.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

data = root / "examples/results/stokes-adaptive"
for nu in (1, 0.01):
    for degree in (0, 1, 2):
        record = json.loads((data / f"polynomial-uniform-nu{nu:g}-g0-l{degree}.json").read_text())
        last = record["rows"][-1]
        print("viscosity", nu, "trace degree", degree, "states", len(record["rows"]),
              "velocity L2", last["velocity_l2"], "pressure L2", last["pressure_l2"],
              "eta1", last["eta1"], "eta2", last["eta2"])


In [ ]:
figures = root / "docs/figures/stokes-adaptive"
display(Image(filename=figures / "published-components.png"))


## Fixed topology versus macro refinement

The threshold is 0.5. Four refinement steps are requested. Red–green macro
closure and rational local-grid closure are specified by the implementation.
The reported effectivity is always $(\eta_1+\eta_2)/\text{mixed error}$;
it is not asserted to exceed one. Printed effectivities in Tables 1–2 are not
universally consistent with that sum, so they are retained as source data rather
than used as equality gates.


In [ ]:
for strategy, degree in (("macro", 0), ("face", 1)):
    name = f"polynomial-{strategy}-nu0.01-g0-l{degree}"
    record = json.loads((data / (name + ".json")).read_text())
    print(strategy, record["stop_reason"], record["rows"][-1])
    display(Image(filename=figures / (name + "-history.png")))
    display(Image(filename=figures / (name + "-fields.png")))


## Smooth cavity variation

The cavity lid is explicitly regularized as $u_x=16x^2(1-x)^2$, $u_y=0$,
with zero velocity on the other walls and mean pressure zero. This is continuous
at the corners; it is not identified with a historical unregularized lid whose
corner convention is unspecified. The cases use viscosity 1 and drag 0 or
$10^4$. Estimator components and divergence alone are not accuracy certificates;
classical conforming Taylor–Hood refinement provides a separate reference.


In [ ]:
for drag in (0, 10000):
    for strategy in ("macro", "face"):
        name = f"cavity-{strategy}-nu1-g{drag:g}-l0"
        record = json.loads((data / (name + ".json")).read_text())
        print(drag, strategy, record["rows"][-1])
        display(Image(filename=figures / (name + "-history.png")))


## Independent conforming reference and physical differences

The classical DOLFINx P2/P1 reference uses the same regularized lid, operator,
viscosity, drag and zero-mean pressure. Seven grids from 8 to 512 squares per
coordinate verify its own refinement sensitivity. The numerical reference is
not an exact solution. The last high-drag energy increment is about 0.395%,
compared with MHM energy differences of 15.9% or more in these finite campaigns.

Differences are integrated on the actual broken MHM fine mesh at Gaussian
orders 12, 20 and 28; all one-sided macro values are preserved. The energy norm
is the square root of the squared velocity-gradient norm plus drag times the
squared velocity L2 norm. Pressure and velocity L2 differences are reported
separately. Reference refinement and MHM errors are distinct measurements.


In [ ]:
display(Image(filename=figures / "cavity-reference-convergence.png"))
for drag in (0, 10000):
    for strategy in ("macro", "face"):
        name = f"cavity-{strategy}-nu1-g{drag:g}-l0-comparison"
        record = json.loads((data / (name + ".json")).read_text())
        print(drag, strategy, record["norms"]["28"])
        display(Image(filename=figures / (name + "-fields.png")))


## Constant lid and fixed corner cutouts

The open top edge has velocity $(1,0)$, with zero velocity on the other edges. The weak MHM trace has no corner smoothing. The DOLFINx sequence uses zero corner nodes and unit interior lid nodes. These data have corner singularities: global pressure and velocity-gradient $L^2$ norms are not finite convergence targets.

Pressure and energy comparisons exclude fixed upper corner squares of side $1/32$, using exact geometric intersections with the fine triangles. Global velocity $L^2$ is reported separately. Both methods use zero mean pressure on the whole square. The classical reference has its own five-level refinement study.

In [ ]:
display(Image(filename=figures / "cavity-constant-reference-convergence.png"))

## Constant-lid adaptive physical fields
The constant lid has corner-singular pressure and velocity gradients. The following comparisons use DOLFINx P2/P1 on a 512×512 grid and a fixed interior domain excluding upper-corner squares of side 1/32. Relative physical norms, reference refinement, and estimator histories are distinct diagnostics. Profiles retain independent one-sided MHM values.


In [ ]:
display(Image(filename=str(figures / "cavity-constant-macro-nu1-g0-l0-longest-edge-theta0.1-comparison-physical-convergence.png")))
display(Image(filename=str(figures / "cavity-constant-macro-nu1-g0-l0-longest-edge-theta0.1-comparison-profiles.png")))


The plateau in the interior norms remains visible as corner refinement increases the number of unknowns. The computed global estimator is not a certified global-energy error bound for this discontinuous boundary trace. Gray profile ticks locate actual macroface intersections.


## Final constant-lid configurations

The following physical differences use the same DOLFINx reference and fixed
corner cutouts. Each record states its threshold, actual fine mesh and stopping
condition. Differences between these configurations are not a cost comparison
at identical approximation spaces. Residuals and refinement indicators are
reported separately from the measured physical field differences.


In [ ]:
configurations = [
    "cavity-constant-macro-nu1-g0-l0-longest-edge-theta0.1",
    "cavity-constant-macro-nu1-g10000-l0-longest-edge-theta0.1-continued",
    "cavity-constant-face-nu1-g0-l0-local-longest-edge-mark-maximum-theta0.1",
    "cavity-constant-face-nu1-g10000-l0-local-longest-edge-mark-maximum",
]
for name in configurations:
    state = json.loads((data / (name + ".json")).read_text())
    comparison = json.loads((data / (name + "-comparison.json")).read_text())
    error = comparison["norms"]["28"]
    print({"case": name, "theta": state["marking_theta"],
           "global_dofs": state["rows"][-1]["trace_and_coarse_dofs"],
           "fine_triangles": state["rows"][-1]["fine_cells"],
           "velocity_percent": 100 * error["velocity_relative"],
           "pressure_percent": 100 * error["pressure_relative"],
           "energy_percent": 100 * error["velocity_energy_relative"]})


In [ ]:
name = configurations[1] + "-comparison"
for suffix in ["upper-layer", "profiles", "physical-convergence"]:
    display(Image(filename=str(figures / f"{name}-{suffix}.png")))
for name in configurations[2:]:
    for suffix in ["physical-convergence", "profiles"]:
        display(Image(filename=str(figures / f"{name}-comparison-{suffix}.png")))
